# M1 — Word TF-IDF + Régression Logistique

Ce modèle utilise TF-IDF pour représenter les énoncés des exercices sous forme de vecteurs, puis une régression logistique pour prédire les huit tags de chaque exercice. La vectorisation repose sur les mots et les expressions présents dans les énoncés. Après l'entraînement, plusieurs configurations d'hyperparamètres sont comparées sur l'ensemble de validation afin de sélectionner la meilleure selon le F1 macro. Les seuils de décision sont ensuite ajustés pour chaque tag, puis le modèle est évalué et sauvegardé. L'ensemble de test reste réservé à l'évaluation finale.

## 1. Charger et contrôler les données

Les descriptions, entrées, sorties et notes forment les seules variables explicatives. Le code, les identifiants et les tags ne sont pas inclus dans le texte.

In [2]:
import hashlib
import html
import json
import platform
import unicodedata
import warnings
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, hamming_loss,
    precision_recall_fscore_support,
)
from sklearn.multiclass import OneVsRestClassifier
from threadpoolctl import threadpool_limits

racine = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "entrainement.jsonl").is_file()), None
)
if racine is None:
    raise FileNotFoundError("Ouvrez ce notebook depuis le dépôt du projet.")

In [3]:
dossier_modele = racine / "models" / "m1_word_tfidf_logistic_regression"
dossier_modele.mkdir(parents=True, exist_ok=True)
graine = 42
tags_cibles = ["math", "graphs", "strings", "number theory",
               "trees", "geometry", "games", "probabilities"]
colonnes_cibles = ["cible_" + tag.replace(" ", "_") for tag in tags_cibles]
champs_texte = {"prob_desc_description": "DESCRIPTION", "prob_desc_input_spec": "ENTREES",
                "prob_desc_output_spec": "SORTIES", "prob_desc_notes": "NOTES"}
versions = {nom: version(nom) for nom in
            ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "joblib", "threadpoolctl"]}
empreintes = {}

def charger_ensemble(nom):
    chemin = racine / "data" / f"{nom}.jsonl"
    contenu = chemin.read_bytes()
    empreintes[nom] = hashlib.sha256(contenu).hexdigest()
    return pd.DataFrame([json.loads(ligne) for ligne in contenu.decode("utf-8").splitlines()
                         if ligne.strip()])

entrainement = charger_ensemble("entrainement")
validation = charger_ensemble("validation")
for donnees in [entrainement, validation]:
    assert donnees[colonnes_cibles].isin([0, 1]).all().all(), "Cibles non binaires."
    assert set(champs_texte).issubset(donnees.columns)
y_train = entrainement[colonnes_cibles].to_numpy(dtype=np.int64)
y_validation = validation[colonnes_cibles].to_numpy(dtype=np.int64)
for champ in ["fichier_source", "src_uid", "code_uid"]:
    assert not entrainement[champ].isna().any() and not validation[champ].isna().any()
    assert not entrainement[champ].duplicated().any()
    assert not validation[champ].duplicated().any()
    assert set(entrainement[champ]).isdisjoint(validation[champ]), f"Fuite : {champ}"
for cibles in [y_train, y_validation]:
    assert ((cibles.sum(axis=0) > 0) & (cibles.sum(axis=0) < len(cibles))).all()

warnings.filterwarnings("error", category=ConvergenceWarning)
print(f"{len(entrainement)} exercices d’entraînement et {len(validation)} de validation.")

3470 exercices d’entraînement et 739 de validation.


## 2. Construire le Word TF-IDF

Pour chaque exercice, la description, les spécifications d’entrée et de sortie et les notes sont réunies en un seul texte. Les champs vides sont ignorés et les caractères HTML et Unicode sont harmonisés. Le TF-IDF transforme ce texte en valeurs numériques à partir des mots et des paires de mots consécutifs. Le vocabulaire et les pondérations sont calculés uniquement sur l’ensemble d’entraînement, puis appliqués à la validation.

In [4]:
def construire_enonce(exercice):
    sections = []
    for champ, titre in champs_texte.items():
        valeur = exercice.get(champ)
        if isinstance(valeur, str) and valeur.strip():
            contenu = unicodedata.normalize("NFC", html.unescape(valeur)).strip()
            sections.append(f"{titre}\n{contenu}")
    return "\n\n".join(sections)

textes_train = entrainement.apply(construire_enonce, axis=1)
textes_validation = validation.apply(construire_enonce, axis=1)
assert textes_train.str.strip().ne("").all() and textes_validation.str.strip().ne("").all()

parametres_tfidf = {
    "ngram_range": (1, 2), "min_df": 2, "max_df": 0.98, "max_features": 80000,
    "sublinear_tf": True, "norm": "l2", "lowercase": True,
    "token_pattern": r"(?u)\b\w+\b",
}
vectoriseur = TfidfVectorizer(**parametres_tfidf)
X_train = vectoriseur.fit_transform(textes_train)
X_validation = vectoriseur.transform(textes_validation)
assert (X_train.getnnz(axis=1) > 0).all()
print(f"{X_train.shape[1]:,} variables TF-IDF.")
display(pd.Series(parametres_tfidf, name="Valeur").to_frame())

71,774 variables TF-IDF.


,Valeur
ngram_range,"(1, 2)"
min_df,2
max_df,0.98
max_features,80000
sublinear_tf,True
norm,l2
lowercase,True
token_pattern,(?u)\b\w+\b


## 3. Comparer six configurations

Huit classifieurs binaires sont entraînés via `OneVsRestClassifier`.
Les six couples `C ∈ {0.1, 1, 10}` et `class_weight ∈ {None, balanced}`
sont comparés avec le seuil par défaut. Le F1 macro donne le même poids
aux huit tags, le F1 micro complète la lecture globale.

In [5]:
def evaluer_predictions(reels, predictions):
    return {
        "f1_macro": float(f1_score(reels, predictions, average="macro", zero_division=0)),
        "f1_micro": float(f1_score(reels, predictions, average="micro", zero_division=0)),
        "hamming_loss": float(hamming_loss(reels, predictions)),
        "exact_match_ratio": float(accuracy_score(reels, predictions)),
    }

In [6]:
seuil_defaut = 0.5

def creer_classifieur(C, ponderation):
    estimateur = LogisticRegression(C=C, solver="liblinear", class_weight=ponderation, max_iter=3000, random_state=graine)
    return OneVsRestClassifier(estimateur, n_jobs=1)

def calculer_scores(classifieur, matrice):
    return classifieur.predict_proba(matrice)

experiences = []
classifieurs = {}
with threadpool_limits(limits=1):
    for C in [0.1, 1.0, 10.0]:
        for nom_ponderation, ponderation in [("aucune", None), ("equilibree", "balanced")]:
            cle = f"C{C}_{nom_ponderation}"
            classifieur = creer_classifieur(C, ponderation)
            debut = perf_counter()
            classifieur.fit(X_train, y_train)
            duree = perf_counter() - debut
            scores = calculer_scores(classifieur, X_validation)
            predictions = (scores >= seuil_defaut).astype(int)
            classifieurs[cle] = classifieur
            experiences.append({"experience": cle, "C": C, "class_weight": nom_ponderation,
                                 "temps_entrainement_s": duree,
                                 **evaluer_predictions(y_validation, predictions)})

recherche = pd.DataFrame(experiences).sort_values(
    ["f1_macro", "C"], ascending=[False, True], kind="stable"
).reset_index(drop=True)
configuration_retenue = recherche.iloc[0]
classifieur_final = classifieurs[configuration_retenue["experience"]]
scores_validation = calculer_scores(classifieur_final, X_validation)
display(recherche.round(4))
print("Configuration retenue :", configuration_retenue["experience"])

,experience,C,class_weight,temps_entrainement_s,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,C1.0_equilibree,1.0,equilibree,0.6166,0.6608,0.6262,0.0666,0.5494
1,C10.0_equilibree,10.0,equilibree,0.9375,0.6445,0.6173,0.0612,0.5846
2,C0.1_equilibree,0.1,equilibree,0.3896,0.6140,0.5724,0.0824,0.4736
3,C10.0_aucune,10.0,aucune,0.7855,0.3997,0.4652,0.0677,0.5494
4,C1.0_aucune,1.0,aucune,0.5467,0.1912,0.2925,0.0753,0.5115
5,C0.1_aucune,0.1,aucune,0.3507,0.0000,0.0000,0.0864,0.4587


Configuration retenue : C1.0_equilibree


## 4. Ajuster les huit seuils

Le classifieur retenu est conservé sans réentraînement. Pour chaque tag, choisir
le seuil donnant le meilleur F1 parmi une grille de quantiles des scores de validation
et le seuil par défaut (0.5).

In [7]:
quantiles_seuils = [0.50, 0.70, 0.80, 0.85, 0.90, 0.925,
                   0.95, 0.965, 0.975, 0.985, 0.99, 0.995]
seuils = []
for indice in range(len(tags_cibles)):
    candidats = np.unique(np.append(
        np.quantile(scores_validation[:, indice], quantiles_seuils), seuil_defaut
    ))
    evaluations = [
        (f1_score(y_validation[:, indice], scores_validation[:, indice] >= seuil,
                  zero_division=0), float(seuil)) for seuil in candidats
    ]
    meilleur = min(evaluations, key=lambda r: (-r[0], abs(r[1] - seuil_defaut), r[1]))
    seuils.append(meilleur[1])
seuils = np.array(seuils)

## 5. Lire les résultats de validation

Comparer les seuils par défaut et ajustés, puis examiner chaque tag avec son support,
sa précision, son rappel et son F1. 

In [8]:
lignes_globales, lignes_tags = [], []
for nom, valeur in [("Aucun tag", 0), ("Tous les tags", 1)]:
    predictions = np.full_like(y_validation, valeur)
    lignes_globales.append({"regle": nom, **evaluer_predictions(
        y_validation, predictions
    )})
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = (scores_validation >= seuils_regle).astype(int)
    lignes_globales.append({"regle": regle,
                           **evaluer_predictions(y_validation, predictions)})
    precisions, rappels, f1, supports = precision_recall_fscore_support(
        y_validation, predictions, average=None, zero_division=0
    )
    for i, tag in enumerate(tags_cibles):
        lignes_tags.append({
            "regle": regle, "tag": tag, "support": int(supports[i]),
            "seuil": float(seuils_regle[i]), "precision": precisions[i],
            "rappel": rappels[i], "f1": f1[i],
        })
bilan = pd.DataFrame(lignes_globales)
par_tag = pd.DataFrame(lignes_tags)
display(bilan.round(4))
display(par_tag.loc[par_tag["regle"].eq("Ajustés")].round(4).reset_index(drop=True))

sans_cible = y_validation.sum(axis=1) == 0
controle_sans_cible = []
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = scores_validation[sans_cible] >= seuils_regle
    avec_prediction = predictions.any(axis=1)
    controle_sans_cible.append({
        "regle": regle,
        "exercices_sans_cible": int(sans_cible.sum()),
        "aucun_tag_predit": int((~avec_prediction).sum()),
        "au_moins_un_tag_predit": int(avec_prediction.sum()),
    })
print("Contrôle des exercices sans tag cible (décomptes) :")
display(pd.DataFrame(controle_sans_cible))

,regle,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,Aucun tag,0.0000,0.0000,0.0864,0.4587
1,Tous les tags,0.1500,0.1591,0.9136,0.0000
2,Défaut,0.6608,0.6262,0.0666,0.5494
3,Ajustés,0.6659,0.6293,0.0660,0.5575


,regle,tag,support,seuil,precision,rappel,f1
0,Ajustés,math,211,0.5000,0.5570,0.6256,0.5893
1,Ajustés,graphs,81,0.5000,0.6463,0.6543,0.6503
2,Ajustés,strings,63,0.5000,0.6389,0.7302,0.6815
3,Ajustés,number theory,52,0.4833,0.5714,0.6154,0.5926
4,Ajustés,trees,49,0.6856,0.7838,0.5918,0.6744
5,Ajustés,geometry,25,0.3757,0.5405,0.8000,0.6452
6,Ajustés,games,16,0.5000,0.9231,0.7500,0.8276
7,Ajustés,probabilities,14,0.5000,1.0000,0.5000,0.6667


Contrôle des exercices sans tag cible (décomptes) :


,regle,exercices_sans_cible,aucun_tag_predit,au_moins_un_tag_predit
0,Défaut,339,225,114
1,Ajustés,339,223,116


## 6. Sauvegarder le modèle final et les résultats

Un unique artefact contient le vectoriseur, les huit classifieurs, les seuils,
l’ordre des tags et les règles de construction du texte. Les trois CSV conservent
la recherche, les métriques globales et le détail par tag. Le modèle sauvegardé
reste celui appris sur l’entraînement, les seuils ont été réglés sur la validation.

In [9]:
modele_final = {
    "modele": "M1", "vectoriseur": vectoriseur, "classifieur": classifieur_final,
    "seuils": seuils, "tags": tags_cibles, "champs_texte": champs_texte,
    "type_scores": "probabilites",
    "pretraitement": {"html_unescape": True, "normalisation_unicode": "NFC",
                      "separateur_sections": "\n\n", "separateur_titre": "\n"},
    "configuration": {"C": float(configuration_retenue["C"]),
                      "class_weight": None if configuration_retenue["class_weight"] == "aucune"
                      else "balanced"},
    "graine": graine, "versions": versions, "python": platform.python_version(),
    "empreintes_donnees": empreintes,
}
_ = joblib.dump(modele_final, dossier_modele / "modele.joblib", compress=3)
recherche.to_csv(dossier_modele / "recherche_hyperparametres.csv", index=False)
bilan.to_csv(dossier_modele / "metriques_validation.csv", index=False)
par_tag.to_csv(dossier_modele / "metriques_par_tag_validation.csv", index=False)